# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AxelYoel/FlyRank-AI-Internship---Axel-Yoel-Chandra/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row = one content item (page), summarized over a 90-day window ending 2026-06-30, applied the same for every client — client history length determines whether that window is full, partial, or empty. Grain (one row per page, no duplicates) is verified via the dim_content grain check.

I chose a 90-day window because 57 clients have 90+ days of GSC history, giving a full window. I also included 10 clients with under 90 days of history, flagged via window_days_available so a reviewer can see how many days of data actually back a given page — if a flagged page looks like it needs review, the reviewer can check its window length and judge whether that's a real signal or just noise from a short history.

I excluded 37 clients entirely because they have no GSC history at all — there's no page-level data to review. Including the 10 partial-history clients rather than excluding them too means losing only ~36% of clients (37/104) from analysis, rather than 45% (37+10) — a cost I judged worth accepting rather than losing more coverage, though I haven't measured the business impact directly.

I chose page-level summary over daily rows because daily position is too noisy on low-traffic days — a small daily sample of searches doesn't reliably represent a page's true position, the same small-sample problem I found in Week 1's CTR analysis. This is verified by comparing position stddev on low- vs. high-impression days: 16.6 vs. 6.3 — noise drops substantially with more searches per day, though it doesn't disappear entirely, confirming daily rows carry real noise even if not pure noise.

In [1]:
%pip -q install duckdb
import duckdb
con = duckdb.connect()

In [2]:
import duckdb, os
from google.colab import userdata

hf_token = userdata.get("HF_token")  # exact name you set in Colab Secrets
os.environ["HF_token"] = hf_token

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

In [3]:
rel = "hf://datasets/FlyRank/internship-warehouse"
con.sql(f"""
    SELECT COUNT(*) AS n_clients,
           MIN(gsc_data_start) AS earliest_client_start,
           MAX(gsc_data_start) AS latest_client_start
    FROM read_parquet('{rel}/dim_clients.parquet')
""").df()

,n_clients,earliest_client_start,latest_client_start
0,104,2025-01-27,2026-06-02


In [4]:
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/dim_clients.parquet')").df()

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,is_active,BOOLEAN,YES,None,None,None
2,has_gsc_access,BOOLEAN,YES,None,None,None
3,has_ga4_access,BOOLEAN,YES,None,None,None
4,access_profile,VARCHAR,YES,None,None,None
5,client_created_date,DATE,YES,None,None,None
6,client_updated_date,DATE,YES,None,None,None
7,gsc_data_start,DATE,YES,None,None,None
8,ga4_data_start,DATE,YES,None,None,None


In [5]:
con.sql(f"""
    SELECT client_hash_id, gsc_data_start,
           DATE_DIFF('day', gsc_data_start, DATE '2026-06-30') AS days_of_history
    FROM read_parquet('{rel}/dim_clients.parquet')
    ORDER BY days_of_history
""").df()

,client_hash_id,gsc_data_start,days_of_history
0,client_aef6ffea193da149,2026-06-02,28
1,client_a22068e339bf95f5,2026-05-24,37
2,client_7de9989c909e91a5,2026-05-18,43
3,client_c7c2962f1c9c3089,2026-05-14,47
4,client_1a8bf67cad4ee525,2026-05-11,50
...,...,...,...
99,client_e921cfa93fbe699b,NaT,<NA>
100,client_f0ff30b229fe9b01,NaT,<NA>
101,client_f4fa4e08a1d500d2,NaT,<NA>
102,client_f63f09ff4e81aa58,NaT,<NA>


In [6]:
con.sql(f"""
    SELECT COUNT(*) AS total_clients,
           SUM(CASE WHEN gsc_data_start IS NULL THEN 1 ELSE 0 END) AS n_null_start
    FROM read_parquet('{rel}/dim_clients.parquet')
""").df()

,total_clients,n_null_start
0,104,37.0


In [7]:
con.sql(f"""
    SELECT
      CASE
        WHEN gsc_data_start IS NULL THEN 'no_gsc_history'
        WHEN days_of_history < 90 THEN '<90 days'
        WHEN days_of_history < 180 THEN '90-180 days'
        WHEN days_of_history < 365 THEN '180-365 days'
        ELSE '365+ days'
      END AS history_bucket,
      COUNT(*) AS n_clients
    FROM (
      SELECT client_hash_id, gsc_data_start,
             DATE_DIFF('day', gsc_data_start, DATE '2026-06-30') AS days_of_history
      FROM read_parquet('{rel}/dim_clients.parquet')
    )
    GROUP BY 1
    ORDER BY 1
""").df()

,history_bucket,n_clients
0,180-365 days,31
1,365+ days,9
2,90-180 days,17
3,<90 days,10
4,no_gsc_history,37


In [8]:
con.sql(f"""
    SELECT has_gsc_access,
           SUM(CASE WHEN gsc_data_start IS NULL THEN 1 ELSE 0 END) AS n_null_start,
           COUNT(*) AS n_clients
    FROM read_parquet('{rel}/dim_clients.parquet')
    GROUP BY 1
""").df()

,has_gsc_access,n_null_start,n_clients
0,False,24.0,27
1,<NA>,6.0,10
2,True,7.0,67


In [9]:
con.sql(f"""
    SELECT client_hash_id, has_gsc_access, gsc_data_start, ga4_data_start, is_active
    FROM read_parquet('{rel}/dim_clients.parquet')
    WHERE has_gsc_access = TRUE AND gsc_data_start IS NULL
""").df()

,client_hash_id,has_gsc_access,gsc_data_start,ga4_data_start,is_active
0,client_04660893ae39614a,True,NaT,2026-05-22,True
1,client_19b89ee4fe3db6da,True,NaT,2026-01-09,False
2,client_5781daf723188fe7,True,NaT,NaT,True
3,client_80ee5b7bd5f4eb89,True,NaT,NaT,True
4,client_91c8eb1698a7b352,True,NaT,NaT,True
5,client_e6d6dce7c2fff733,True,NaT,NaT,True
6,client_f0ff30b229fe9b01,True,NaT,NaT,True


In [10]:
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/dim_content.parquet')").df()

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


In [11]:
con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(DISTINCT content_hash_id) AS unique_content_ids
    FROM read_parquet('{rel}/dim_content.parquet')
""").df()

,total_rows,unique_content_ids
0,519606,519606


In [12]:
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [13]:
con.sql(f"""
    SELECT content_hash_id, report_date, gsc_avg_position, gsc_clicks, gsc_impressions
    FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
    WHERE content_hash_id IN (
        SELECT content_hash_id
        FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
        WHERE gsc_data_available = TRUE AND gsc_impressions > 0
        GROUP BY 1 HAVING COUNT(*) > 20
        LIMIT 3
    )
    ORDER BY content_hash_id, report_date
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,report_date,gsc_avg_position,gsc_clicks,gsc_impressions
0,content_48a102a5060ed0e2,2026-06-01,52.000000,0,3
1,content_48a102a5060ed0e2,2026-06-02,29.500000,0,4
2,content_48a102a5060ed0e2,2026-06-03,28.333333,0,3
3,content_48a102a5060ed0e2,2026-06-04,32.400000,0,5
4,content_48a102a5060ed0e2,2026-06-05,NaN,0,0
...,...,...,...,...,...
85,content_bf89a688c0ec7cd7,2026-06-26,89.000000,0,1
86,content_bf89a688c0ec7cd7,2026-06-27,66.000000,0,3
87,content_bf89a688c0ec7cd7,2026-06-28,81.666667,0,3
88,content_bf89a688c0ec7cd7,2026-06-29,85.333333,0,3


In [14]:
con.sql(f"""
    SELECT AVG(pos_std) AS avg_within_page_position_stddev
    FROM (
      SELECT content_hash_id, STDDEV(gsc_avg_position) AS pos_std
      FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
      WHERE gsc_data_available = TRUE AND gsc_avg_position > 0
      GROUP BY 1
      HAVING COUNT(*) > 10
    )
""").df()

,avg_within_page_position_stddev
0,11.378559


In [15]:
con.sql(f"""
    SELECT
      CASE WHEN gsc_impressions < 10 THEN 'low_impr_days' ELSE 'higher_impr_days' END AS impr_bucket,
      AVG(pos_std) AS avg_position_stddev
    FROM (
      SELECT content_hash_id,
             AVG(gsc_impressions) AS gsc_impressions,
             STDDEV(gsc_avg_position) AS pos_std
      FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
      WHERE gsc_data_available = TRUE AND gsc_avg_position > 0
      GROUP BY 1
      HAVING COUNT(*) > 10
    )
    GROUP BY 1
""").df()

,impr_bucket,avg_position_stddev
0,low_impr_days,16.571946
1,higher_impr_days,6.320960


## 2. Field classification

Lane 4 works in two stages, and that shapes every bucket below. Stage 1 builds the label directly — each page's actual CTR compared against its position tier's expected CTR, a formula, not a model. Stage 2 trains a classifier on that label. Anything used to build the label in Stage 1 can't also be a Stage 2 feature, or the model would just be handed its own answer.

### Label ingredients

- `gsc_clicks`, `gsc_impressions`, `gsc_avg_position` are never features. I use them only to compute each page's actual CTR and its `position_tier`, then compare that CTR to its tier's median to get the label. They can't appear on both sides of the problem.

### Context

- IDs across all four tables (`client_hash_id`, `content_hash_id`, `keyword_hash_id`, `url_hash_id`, `query_hash_id`) are for joins and grouping only — the values themselves carry no information.
- `gsc_data_start`, `ga4_data_start`, `window_start`, `window_end` define time windows rather than predict anything.
- `is_published`/`is_deleted` decide which rows I analyze in the first place, not what a model sees.
- `gsc_data_available`/`ga4_data_available`/`client_has_gsc`/`client_has_ga4` are context too, but I treat them carefully — they're three-valued (True/False/NULL), so I filter with `IS TRUE`/`IS NOT TRUE`, never `= TRUE`, or NULL rows disappear silently.
- `content_created_date`/`content_updated_date`/`keyword_created_date` stay as context rather than features, since I only use them to derive age/freshness — keeping both the raw date and its derived number would just be the same information twice.
- `engaged_sessions`/`total_engagement_sec` stay as context for the same reason: 93.5% of GA4-tracked rows have zero engaged sessions, so a ratio of the two is undefined most of the time — I use this pair only to build a simpler "had engagement" flag instead.
- `gsc_sum_position` is redundant with `gsc_avg_position`, so it stays as context too.

### Excluded

- `has_gsc_access`/`has_ga4_access` turned out unreliable when I checked them — 7 clients show `has_gsc_access = True` with no `gsc_data_start` at all, and I couldn't find a pattern explaining it, so I use `gsc_data_start IS NOT NULL` instead of trusting the flag.
- `provider_used`/`model_used` reflect a client's business choice, not the content's actual quality, so I excluded them.
- `optimization_eligible_date` looked suspicious when I checked it — the dates cluster into batches rather than following a fixed offset from creation, and I couldn't rule out a link to performance with the data available, so I excluded it and noted the ambiguity rather than guessing.
- `month` I excluded because feeding it in as a raw number would tell a model December is "worth more" than January, which isn't true.
- `ga4_sessions` doesn't reconcile with the sessions-source columns at the row level — individual sources can exceed the total — and nothing documents why, so I excluded it and kept the individual source columns instead, since each one is at least internally consistent on its own.
- `pageviews`/`users` I excluded because they're most likely a consequence of ranking well, not a cause of it — using them risks the model learning to predict good performance from a symptom of good performance.
- `impressions_90d`/`clicks_90d`/`avg_position_90d`/`impressions_last30`/`clicks_last30`/`avg_position_last30`/`content_total_impressions_90d` (query table) are all excluded because that table's window (2026-04-02 to 2026-06-30) lines up with my label's own window — none of that data would have been known before the label period, so using it would be leakage.
- `position_tier` is excluded from Stage 2 for the same reason — it's part of what defines the label in Stage 1, so feeding it back in would leak that construction straight into the model.

### Features

- Content metadata: `content_type`, `main_intent`, `competition_level`, `backlinks`, `category_count`, `search_volume`, `competition`, `cpc`, `char_count`, `word_count`, `keyword_token_count`, `keyword_char_count`, plus derived content age, freshness, and keyword age.
- Traffic-source breakdown: `sessions_ai`, `sessions_organic`, `sessions_direct`, `sessions_referral`, `sessions_social`, `sessions_paid` — I keep these as individual signals rather than assuming they sum to a total, since they don't reconcile cleanly against `ga4_sessions`.
- `scroll_events`, plus the derived "had engagement" flag.
- `impressions_prev30`, `clicks_prev30`, `avg_position_prev30` are safe because May data was already fully known before my June label window even started.
- `content_visible_query_count`, `rare_impressions_share`, `query_char_count`, `query_token_count` round these out.
- `rare_query_count` and `anonymized_impressions_share` I'm including too, but I haven't been able to confirm their exact definitions, so I'm treating them cautiously rather than fully trusting them.
- All of the query-table features need aggregating up from query-level to page-level first, since that table's grain is finer than my page-level unit of analysis.

In [16]:
con.sql(f"""
    SELECT content_created_date, optimization_eligible_date,
           DATE_DIFF('day', content_created_date, optimization_eligible_date) AS days_to_eligible
    FROM read_parquet('{rel}/dim_content.parquet')
    WHERE optimization_eligible_date IS NOT NULL
    LIMIT 20
""").df()

,content_created_date,optimization_eligible_date,days_to_eligible
0,2026-04-16,2026-08-07,113
1,2026-04-18,2026-08-07,111
2,2026-05-08,2026-07-30,83
3,2026-04-16,2026-08-07,113
4,2026-05-20,2026-07-30,71
5,2026-04-25,2026-07-26,92
6,2026-05-09,2026-07-30,82
7,2026-05-21,2026-08-08,79
8,2026-05-16,2026-07-30,75
9,2026-04-24,2026-07-26,93


In [17]:
con.sql(f"""
    SELECT
      SUM(sessions_ai) AS total_sessions_ai,
      SUM(ai_chatgpt + ai_perplexity + ai_gemini + ai_copilot + ai_claude + ai_meta + ai_other) AS sum_of_ai_breakdown,
      SUM(sessions_ai) - SUM(ai_chatgpt + ai_perplexity + ai_gemini + ai_copilot + ai_claude + ai_meta + ai_other) AS difference
    FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_sessions_ai,sum_of_ai_breakdown,difference
0,32762.0,32776.0,-14.0


In [18]:
con.sql(f"""
    SELECT
      SUM(CASE WHEN ai_chatgpt IS NULL THEN 1 ELSE 0 END) AS null_chatgpt,
      SUM(CASE WHEN ai_perplexity IS NULL THEN 1 ELSE 0 END) AS null_perplexity,
      SUM(CASE WHEN ai_gemini IS NULL THEN 1 ELSE 0 END) AS null_gemini,
      SUM(CASE WHEN ai_copilot IS NULL THEN 1 ELSE 0 END) AS null_copilot,
      SUM(CASE WHEN ai_claude IS NULL THEN 1 ELSE 0 END) AS null_claude,
      SUM(CASE WHEN ai_meta IS NULL THEN 1 ELSE 0 END) AS null_meta,
      SUM(CASE WHEN ai_other IS NULL THEN 1 ELSE 0 END) AS null_other
    FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
""").df()

,null_chatgpt,null_perplexity,null_gemini,null_copilot,null_claude,null_meta,null_other
0,2397428.0,2397428.0,2397428.0,2397428.0,2397428.0,2397428.0,2397428.0


In [19]:
con.sql(f"""
    SELECT COUNT(*) AS total_rows
    FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
""").df()

,total_rows
0,11694072


In [20]:
con.sql(f"""
    SELECT
      CASE WHEN sessions_ai IS NULL THEN 'sessions_ai_null'
           WHEN sessions_ai = 0 THEN 'sessions_ai_zero'
           ELSE 'sessions_ai_positive' END AS sessions_ai_status,
      COUNT(*) AS n_rows,
      SUM(CASE WHEN ai_chatgpt IS NULL THEN 1 ELSE 0 END) AS null_chatgpt_in_group
    FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
    GROUP BY 1
""").df()

,sessions_ai_status,n_rows,null_chatgpt_in_group
0,sessions_ai_null,2397428,2397428.0
1,sessions_ai_zero,9291351,0.0
2,sessions_ai_positive,5293,0.0


In [21]:
con.sql(f"""
    SELECT
      SUM(ga4_sessions) AS total_ga4_sessions,
      SUM(sessions_organic + sessions_direct + sessions_referral + sessions_social + sessions_paid + sessions_ai) AS sum_of_breakdown,
      SUM(ga4_sessions) - SUM(sessions_organic + sessions_direct + sessions_referral + sessions_social + sessions_paid + sessions_ai) AS difference
    FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_ga4_sessions,sum_of_breakdown,difference
0,2759763.0,3770527.0,-1010764.0


In [22]:
con.sql(f"""
    SELECT
      SUM(CASE WHEN sessions_organic IS NULL THEN 1 ELSE 0 END) AS null_organic,
      SUM(CASE WHEN sessions_direct IS NULL THEN 1 ELSE 0 END) AS null_direct,
      SUM(CASE WHEN sessions_referral IS NULL THEN 1 ELSE 0 END) AS null_referral,
      SUM(CASE WHEN sessions_social IS NULL THEN 1 ELSE 0 END) AS null_social,
      SUM(CASE WHEN sessions_paid IS NULL THEN 1 ELSE 0 END) AS null_paid,
      SUM(CASE WHEN ga4_sessions IS NULL THEN 1 ELSE 0 END) AS null_ga4_sessions
    FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
""").df()

,null_organic,null_direct,null_referral,null_social,null_paid,null_ga4_sessions
0,2397428.0,2397428.0,2397428.0,2397428.0,2397428.0,2397428.0


In [23]:
con.sql(f"""
    SELECT ga4_sessions, sessions_organic, sessions_direct, sessions_referral,
           sessions_social, sessions_paid, sessions_ai,
           (sessions_organic + sessions_direct + sessions_referral + sessions_social + sessions_paid + sessions_ai) AS breakdown_sum
    FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
    WHERE ga4_sessions IS NOT NULL AND ga4_sessions > 0
    LIMIT 15
""").df()

,ga4_sessions,sessions_organic,sessions_direct,sessions_referral,sessions_social,sessions_paid,sessions_ai,breakdown_sum
0,1,2,0,0,0,0,0,2
1,1,2,0,0,0,0,0,2
2,1,2,0,0,0,0,0,2
3,1,2,0,0,0,0,0,2
4,1,0,0,0,0,0,0,0
5,1,0,1,0,0,0,0,1
6,1,2,0,0,0,0,0,2
7,1,2,0,0,0,0,0,2
8,2,0,2,0,0,0,0,2
9,2,2,0,0,0,0,0,2


In [24]:
con.sql(f"""
    SELECT
      SUM(CASE WHEN ga4_engaged_sessions = 0 THEN 1 ELSE 0 END) AS zero_engaged_rows,
      COUNT(*) AS total_rows
    FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,zero_engaged_rows,total_rows
0,9255116.0,11694072


In [25]:
con.sql(f"""
    SELECT ga4_data_available,
           COUNT(*) AS n_rows,
           SUM(CASE WHEN ga4_engaged_sessions = 0 THEN 1 ELSE 0 END) AS n_zero_engaged
    FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
    GROUP BY 1
""").df()

,ga4_data_available,n_rows,n_zero_engaged
0,False,8651918,8651918.0
1,<NA>,2397428,0.0
2,True,644726,603198.0


In [26]:
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_query_90d.parquet')").df()

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,query_hash_id,VARCHAR,YES,None,None,None
3,query_char_count,BIGINT,YES,None,None,None
4,query_token_count,BIGINT,YES,None,None,None
5,window_start,DATE,YES,None,None,None
6,window_end,DATE,YES,None,None,None
7,impressions_90d,BIGINT,YES,None,None,None
8,clicks_90d,BIGINT,YES,None,None,None
9,impressions_last30,BIGINT,YES,None,None,None


In [27]:
con.sql(f"""
    SELECT MIN(window_start) AS earliest_start, MAX(window_end) AS latest_end
    FROM read_parquet('{rel}/fact_content_query_90d.parquet')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,earliest_start,latest_end
0,2026-04-02,2026-06-30


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [30]:
# --- Section 1, Claim 1: grain check — one row per page in dim_content ---
con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(DISTINCT content_hash_id) AS unique_content_ids
    FROM read_parquet('{rel}/dim_content.parquet')
""").df()

,total_rows,unique_content_ids
0,519606,519606


In [31]:
# --- Section 1, Claim 2: client history depth — why per-client window handling matters ---
con.sql(f"""
    SELECT
      CASE
        WHEN gsc_data_start IS NULL THEN 'no_gsc_history'
        WHEN days_of_history < 90 THEN '<90 days'
        WHEN days_of_history < 180 THEN '90-180 days'
        WHEN days_of_history < 365 THEN '180-365 days'
        ELSE '365+ days'
      END AS history_bucket,
      COUNT(*) AS n_clients
    FROM (
      SELECT client_hash_id, gsc_data_start,
             DATE_DIFF('day', gsc_data_start, DATE '2026-06-30') AS days_of_history
      FROM read_parquet('{rel}/dim_clients.parquet')
    )
    GROUP BY 1
    ORDER BY 1
""").df()

,history_bucket,n_clients
0,180-365 days,31
1,365+ days,9
2,90-180 days,17
3,<90 days,10
4,no_gsc_history,37


In [32]:
# --- Section 1, Claim 3: daily position noise, low- vs high-impression days ---
con.sql(f"""
    SELECT
      CASE WHEN gsc_impressions < 10 THEN 'low_impr_days' ELSE 'higher_impr_days' END AS impr_bucket,
      AVG(pos_std) AS avg_position_stddev
    FROM (
      SELECT content_hash_id,
             AVG(gsc_impressions) AS gsc_impressions,
             STDDEV(gsc_avg_position) AS pos_std
      FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
      WHERE gsc_data_available = TRUE AND gsc_avg_position > 0
      GROUP BY 1
      HAVING COUNT(*) > 10
    )
    GROUP BY 1
""").df()

,impr_bucket,avg_position_stddev
0,low_impr_days,16.571946
1,higher_impr_days,6.320960


In [33]:
# --- Section 2, Claim 4: has_gsc_access is not fully trustworthy vs gsc_data_start ---
con.sql(f"""
    SELECT has_gsc_access,
           SUM(CASE WHEN gsc_data_start IS NULL THEN 1 ELSE 0 END) AS n_null_start,
           COUNT(*) AS n_clients
    FROM read_parquet('{rel}/dim_clients.parquet')
    GROUP BY 1
""").df()

,has_gsc_access,n_null_start,n_clients
0,False,24.0,27
1,<NA>,6.0,10
2,True,7.0,67


In [34]:
# --- Section 2, Claim 5: sessions_ai reconciles near-exactly with the 6-provider breakdown ---
con.sql(f"""
    SELECT
      SUM(sessions_ai) AS total_sessions_ai,
      SUM(ai_chatgpt + ai_perplexity + ai_gemini + ai_copilot + ai_claude + ai_meta + ai_other) AS sum_of_ai_breakdown,
      SUM(sessions_ai) - SUM(ai_chatgpt + ai_perplexity + ai_gemini + ai_copilot + ai_claude + ai_meta + ai_other) AS difference
    FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
""").df()

# NULL pattern: sessions_ai and its breakdown are missing together, not randomly
con.sql(f"""
    SELECT
      CASE WHEN sessions_ai IS NULL THEN 'sessions_ai_null'
           WHEN sessions_ai = 0 THEN 'sessions_ai_zero'
           ELSE 'sessions_ai_positive' END AS sessions_ai_status,
      COUNT(*) AS n_rows,
      SUM(CASE WHEN ai_chatgpt IS NULL THEN 1 ELSE 0 END) AS null_chatgpt_in_group
    FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
    GROUP BY 1
""").df()

,sessions_ai_status,n_rows,null_chatgpt_in_group
0,sessions_ai_null,2397428,2397428.0
1,sessions_ai_zero,9291351,0.0
2,sessions_ai_positive,5293,0.0


In [35]:
# --- Section 2, Claim 6: ga4_sessions does NOT reconcile with the sessions-source breakdown ---
con.sql(f"""
    SELECT
      SUM(ga4_sessions) AS total_ga4_sessions,
      SUM(sessions_organic + sessions_direct + sessions_referral + sessions_social + sessions_paid + sessions_ai) AS sum_of_breakdown,
      SUM(ga4_sessions) - SUM(sessions_organic + sessions_direct + sessions_referral + sessions_social + sessions_paid + sessions_ai) AS difference
    FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
""").df()

,total_ga4_sessions,sum_of_breakdown,difference
0,2759763.0,3770527.0,-1010764.0


In [37]:
# --- Section 2, Claim 7: engaged_sessions is zero on the vast majority of GA4-tracked rows ---
con.sql(f"""
    SELECT ga4_data_available,
           COUNT(*) AS n_rows,
           SUM(CASE WHEN ga4_engaged_sessions = 0 THEN 1 ELSE 0 END) AS n_zero_engaged
    FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
    GROUP BY 1
""").df()

,ga4_data_available,n_rows,n_zero_engaged
0,False,8651918,8651918.0
1,<NA>,2397428,0.0
2,True,644726,603198.0


In [38]:
# --- Section 2, Claim 8: fact_content_query_90d window overlaps the label window ---
con.sql(f"""
    SELECT MIN(window_start) AS earliest_start, MAX(window_end) AS latest_end
    FROM read_parquet('{rel}/fact_content_query_90d.parquet')
""").df()

,earliest_start,latest_end
0,2026-04-02,2026-06-30


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [29]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.